# Zürich's drinking fountains: GIS analysis with the Overpass Web API

Zürich has more than 1200 public drinking fountains, most of them run by the city's water utility (WVZ). In this notebook we:

1. Query **points** (fountains) and **polygons** (city quarters) from OpenStreetMap using the [Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API)
2. Build a **GeoJSON** file by hand
3. Transform coordinates from WGS84 to the Swiss coordinate system **LV95 (EPSG:2056)**
4. Run spatial analyses: **point-in-polygon**, **areas**, **densities** and **distance to the nearest fountain**
5. Visualize the results with interactive **web maps** (folium) and **charts** (plotly)

Tip: the queries below can also be tested interactively on [overpass-turbo.eu](https://overpass-turbo.eu/).

## Libraries and settings

In [ ]:
# Libraries
import os
import json
import time

import requests
import numpy as np
import pandas as pd
import folium
from folium import plugins
import plotly.express as px
import matplotlib.pyplot as plt
from matplotlib.path import Path

# Colors: one fixed color per water source
WATER_COLORS = {'Spring water': '#2a78d6',
                'Tap water': '#eb6834',
                'Unknown': '#9a9994'}

# Swisstopo base maps (free to use, see https://www.geo.admin.ch/)
def swisstopo_url(layer, ext='jpeg'):
    """Tile URL of a swisstopo WMTS layer in Web Mercator (EPSG:3857)."""
    return (f'https://wmts.geo.admin.ch/1.0.0/{layer}/default/current/3857/'
            f'{{z}}/{{x}}/{{y}}.{ext}')


SWISSTOPO_ATTR = '&copy; <a href="https://www.swisstopo.admin.ch/">swisstopo</a>'

# Current working directory
print(f'Current working directory: {os.getcwd()}')

## Overpass API helper function

We use the Swiss Overpass server [overpass.osm.ch](https://overpass.osm.ch/), which contains OpenStreetMap data for Switzerland. Public Overpass servers are often busy, so the function retries a failed request and waits longer after each failed attempt (exponential backoff).

In [ ]:
# Swiss Overpass API endpoint
OVERPASS_URL = 'https://overpass.osm.ch/api/interpreter'


def overpass_query(query, max_attempts=6, wait_time=10):
    """Send a query to the Swiss Overpass API and return the list of elements."""
    for attempt in range(1, max_attempts + 1):
        try:
            r = requests.post(OVERPASS_URL, data={'data': query}, timeout=120)
            r.raise_for_status()
            elements = r.json()['elements']
            print(f'Attempt {attempt}: success, {len(elements)} elements received.')
            return elements
        except (requests.exceptions.RequestException, ValueError, KeyError) as e:
            print(f'Attempt {attempt} failed: {e}')
            if attempt < max_attempts:
                print(f'Waiting {wait_time} seconds before retrying ...')
                time.sleep(wait_time)
                wait_time *= 2
    raise RuntimeError('Overpass API not available after several attempts.')

## Get data

### Points: drinking fountains

OSM tag [`amenity=drinking_water`](https://wiki.openstreetmap.org/wiki/Tag:amenity%3Ddrinking_water). `nwr` searches nodes, ways and relations; `out center` returns a single point for ways and relations as well.

In [ ]:
query = """
[out:json][timeout:90];
area["name"="Zürich"]["admin_level"="8"]->.searchArea;
nwr["amenity"="drinking_water"](area.searchArea);
out center;
"""

fountains = overpass_query(query)

# Save raw data to file
with open('drinking_fountains.json', 'w') as f:
    json.dump(fountains, f)

# Example of a single element
fountains[0]

In [ ]:
# Flatten the nested JSON into a data frame
df_raw = pd.json_normalize(fountains)

# Ways have no lat/lon, but a center point
if 'center.lat' in df_raw:
    df_raw['lat'] = df_raw['lat'].fillna(df_raw['center.lat'])
    df_raw['lon'] = df_raw['lon'].fillna(df_raw['center.lon'])

# Select and rename relevant columns
columns = {
    'id': 'osm_id',
    'type': 'osm_type',
    'lat': 'lat',
    'lon': 'lon',
    'tags.name': 'name',
    'tags.drinking_water:description': 'water',
    'tags.fountain': 'fountain_type',
    'tags.material': 'material',
    'tags.bottle': 'bottle',
    'tags.dog': 'dog',
    'tags.wheelchair': 'wheelchair',
    'tags.operator': 'operator',
    'tags.artist_name': 'artist',
    'tags.start_date': 'start_date'}
df = df_raw.reindex(columns=columns.keys()).rename(columns=columns)

# Translate the water source (German in OSM) into three categories
df['water_source'] = (df['water']
                      .map({'Quellwasser': 'Spring water', 'Leitungswasser': 'Tap water'})
                      .fillna('Unknown'))

print(df.shape)
df.head()

### Polygons: city quarters

The 34 city quarters of Zürich are boundary relations with `admin_level=10`. `out geom` returns the coordinates of all member ways, which we join ("stitch") into closed rings.

In [ ]:
query = """
[out:json][timeout:120];
area["name"="Zürich"]["admin_level"="8"]->.searchArea;
rel["boundary"="administrative"]["admin_level"="10"](area.searchArea);
out geom;
"""

quarters = overpass_query(query)

In [ ]:
def stitch_rings(ways):
    """Join a list of ways (lists of (lon, lat) points) into closed rings."""
    ways = [list(w) for w in ways]
    rings = []
    while ways:
        ring = ways.pop(0)
        while ring[0] != ring[-1]:
            for i, way in enumerate(ways):
                if way[0] == ring[-1]:
                    ring += way[1:]
                    break
                if way[-1] == ring[-1]:
                    ring += way[::-1][1:]
                    break
            else:
                break  # ring cannot be closed
            ways.pop(i)
        rings.append(ring)
    return rings


# Build a GeoJSON FeatureCollection by hand
features = []
for rel in quarters:
    outer_ways = [[(p['lon'], p['lat']) for p in m['geometry']]
                  for m in rel['members']
                  if m['type'] == 'way' and m['role'] == 'outer']
    rings = stitch_rings(outer_ways)
    features.append({
        'type': 'Feature',
        'properties': {'name': rel['tags']['name']},
        'geometry': {'type': 'MultiPolygon',
                     'coordinates': [[ring] for ring in rings]}})

geojson = {'type': 'FeatureCollection', 'features': features}

# Save as GeoJSON file (can be opened in QGIS or on geojson.io)
with open('zurich_quarters.geojson', 'w') as f:
    json.dump(geojson, f)

print(f'{len(features)} quarters:', ', '.join(sorted(f["properties"]["name"] for f in features)))

### Polygon: Lake Zürich

The city boundary includes part of the lake. We download the lake polygon so we can exclude the water surface from the analysis later.

In [ ]:
query = """
[out:json][timeout:120];
rel["natural"="water"]["name"="Zürichsee"];
out geom;
"""

lake = overpass_query(query)[0]
lake_rings = stitch_rings([[(p['lon'], p['lat']) for p in m['geometry']]
                           for m in lake['members']
                           if m['type'] == 'way' and m['role'] == 'outer'])
print(f'{lake["tags"]["name:en"]}: {len(lake_rings)} outer ring(s)')

## Coordinate transformation: WGS84 to LV95

Longitude/latitude are angles, so we cannot use them directly for distances or areas. We transform them into the Swiss coordinate system LV95 (east/north in meters), using swisstopo's [approximate formulas](https://www.swisstopo.admin.ch/en/transformation-calculation-services) (accuracy: about 1 m).

In [ ]:
def wgs84_to_lv95(lon, lat):
    """Transform WGS84 longitude/latitude (degrees) into LV95 east/north (meters)."""
    lon, lat = np.asarray(lon, dtype=float), np.asarray(lat, dtype=float)
    phi = (lat * 3600 - 169028.66) / 10000
    lam = (lon * 3600 - 26782.5) / 10000
    east = (2600072.37 + 211455.93 * lam - 10938.51 * lam * phi
            - 0.36 * lam * phi**2 - 44.54 * lam**3)
    north = (1200147.07 + 308807.95 * phi + 3745.25 * lam**2
             + 76.63 * phi**2 - 194.56 * lam**2 * phi + 119.79 * phi**3)
    return east, north


# Test: Zürich main station
print(wgs84_to_lv95(8.5402, 47.3782))

df['east'], df['north'] = wgs84_to_lv95(df['lon'], df['lat'])
df[['name', 'lon', 'lat', 'east', 'north']].head()

## Spatial analysis

### Vector analysis: area and point-in-polygon

- **Area**: shoelace formula on the LV95 coordinates
- **Point-in-polygon**: `matplotlib.path.Path.contains_points()` tests which fountains lie inside a quarter

In [ ]:
def shoelace_area(x, y):
    """Area of a polygon (in the unit of the coordinates squared)."""
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))


points = df[['east', 'north']].to_numpy()
df['quarter'] = None
paths = {}
rows = []

for feature in geojson['features']:
    name = feature['properties']['name']
    area = 0
    inside = np.zeros(len(df), dtype=bool)
    paths[name] = []
    for polygon in feature['geometry']['coordinates']:
        lon, lat = np.array(polygon[0]).T
        east, north = wgs84_to_lv95(lon, lat)
        area += shoelace_area(east, north)
        path = Path(np.column_stack([east, north]))
        paths[name].append(path)
        inside |= path.contains_points(points)
    df.loc[inside, 'quarter'] = name
    rows.append({'quarter': name, 'area_km2': area / 1e6, 'fountains': inside.sum()})

df_quarters = pd.DataFrame(rows).set_index('quarter')

print(f'Total area of all quarters: {df_quarters.area_km2.sum():.1f} km² (including part of the lake)')
print(f'Fountains assigned to a quarter: {df["quarter"].notna().sum()} of {len(df)}')
df_quarters.sort_values('fountains', ascending=False).head().round(2)

### Raster analysis: land area and density per quarter

We cover the city with a 50 m grid (raster). Each grid cell is assigned to a quarter, and cells on the lake are removed. Counting the remaining cells gives the **land area** of each quarter.

In [ ]:
cell = 50  # grid cell size in meters

# Grid over the bounding box of the city
all_paths = [p for ps in paths.values() for p in ps]
vertices = np.vstack([p.vertices for p in all_paths])
xs = np.arange(vertices[:, 0].min(), vertices[:, 0].max(), cell)
ys = np.arange(vertices[:, 1].min(), vertices[:, 1].max(), cell)
grid_x, grid_y = np.meshgrid(xs, ys)
grid = np.column_stack([grid_x.ravel(), grid_y.ravel()])

# Grid cells on the lake
lake_paths = [Path(np.column_stack(wgs84_to_lv95(*np.array(ring).T))) for ring in lake_rings]
on_lake = np.zeros(len(grid), dtype=bool)
for p in lake_paths:
    on_lake |= p.contains_points(grid)

# Assign land cells to quarters
grid_quarter = np.full(len(grid), None, dtype=object)
for name, ps in paths.items():
    for p in ps:
        grid_quarter[p.contains_points(grid) & ~on_lake] = name
in_city = grid_quarter != None  # noqa: E711

df_quarters['land_km2'] = (pd.Series(grid_quarter[in_city]).value_counts()
                           * cell**2 / 1e6)
df_quarters['per_km2'] = df_quarters['fountains'] / df_quarters['land_km2']
df_quarters = df_quarters.sort_values('per_km2', ascending=False)

print(f'Land area of the city: {df_quarters.land_km2.sum():.1f} km² '
      f'(lake: {df_quarters.area_km2.sum() - df_quarters.land_km2.sum():.1f} km²)')
df_quarters.head(10).round(2)

### Distance to the nearest fountain

For every land cell of the grid we compute the straight-line distance to the nearest fountain.

In [ ]:
# Nearest fountain distance (in chunks to save memory)
dist = np.full(len(grid), np.nan)
idx = np.where(in_city)[0]
for chunk in np.array_split(idx, 20):
    d = np.hypot(grid[chunk, None, 0] - points[None, :, 0],
                 grid[chunk, None, 1] - points[None, :, 1])
    dist[chunk] = d.min(axis=1)
dist_grid = dist.reshape(grid_x.shape)

d = dist[in_city]
print(f'Grid cells in the city: {in_city.sum():,}')
print(f'Median distance to the nearest fountain: {np.median(d):.0f} m')
for r in [100, 200, 300, 500]:
    print(f'Share of the city within {r:>3} m of a fountain: {(d <= r).mean():.1%}')

## Visualize data

### What kind of fountains are there?

In [ ]:
# Water source and fountain type
df_types = (df.assign(fountain_type=df['fountain_type'].fillna('unknown'))
            .groupby(['fountain_type', 'water_source']).size()
            .reset_index(name='count'))

fig = px.bar(df_types, x='count', y='fountain_type', color='water_source',
             orientation='h', color_discrete_map=WATER_COLORS,
             category_orders={'water_source': list(WATER_COLORS)},
             labels={'count': 'Number of fountains', 'fountain_type': 'Fountain type',
                     'water_source': 'Water source'},
             title='Fountain types and water sources in Zürich',
             template='simple_white', height=380)
fig.update_yaxes(categoryorder='total ascending')
fig.show()

In [ ]:
# How complete is OpenStreetMap? Share of fountains with a value per attribute
attributes = ['name', 'water', 'fountain_type', 'material', 'bottle', 'dog',
              'wheelchair', 'operator', 'artist', 'start_date']
completeness = (df[attributes].notna().mean() * 100).sort_values().reset_index()
completeness.columns = ['attribute', 'share']

fig = px.bar(completeness, x='share', y='attribute', orientation='h',
             text=completeness['share'].map('{:.0f}%'.format),
             labels={'share': 'Fountains with this attribute (%)', 'attribute': 'OSM attribute'},
             title='Data completeness: how many fountains have each attribute?',
             template='simple_white', height=420)
fig.update_traces(marker_color='#2a78d6', textposition='outside')
fig.update_xaxes(range=[0, 110])
fig.show()

### Interactive fountain map

Use the layer control (top right) to switch between base maps (swisstopo light map, national map, aerial image, OpenStreetMap) and layers (clusters per water source, heatmap). Click on a fountain for details.

In [ ]:
def popup_html(row):
    """HTML table with the attributes of a fountain."""
    fields = {'Name': row['name'], 'Water': row['water_source'],
              'Type': row['fountain_type'], 'Material': row['material'],
              'Bottle refill': row['bottle'], 'Dog bowl': row['dog'],
              'Wheelchair': row['wheelchair'], 'Artist': row['artist'],
              'Built': row['start_date'], 'Quarter': row['quarter']}
    rows = ''.join(f'<tr><td><b>{k}</b></td><td>{v}</td></tr>'
                   for k, v in fields.items() if pd.notna(v))
    link = f'https://www.openstreetmap.org/{row["osm_type"]}/{row["osm_id"]}'
    return f'<table>{rows}</table><a href="{link}" target="_blank">View on OSM</a>'


m = folium.Map(location=[df.lat.mean(), df.lon.mean()], zoom_start=13,
               tiles=None, control_scale=True)

# Base maps
folium.TileLayer(swisstopo_url('ch.swisstopo.swisstlm3d-karte-grau', 'png'),
                 attr=SWISSTOPO_ATTR, name='swisstopo light map').add_to(m)
folium.TileLayer(swisstopo_url('ch.swisstopo.pixelkarte-farbe'),
                 attr=SWISSTOPO_ATTR, name='swisstopo national map', show=False).add_to(m)
folium.TileLayer(swisstopo_url('ch.swisstopo.swissimage'),
                 attr=SWISSTOPO_ATTR, name='swisstopo aerial image', show=False).add_to(m)
folium.TileLayer('OpenStreetMap', name='OpenStreetMap', show=False).add_to(m)

# Quarter boundaries
folium.GeoJson(geojson, name='City quarters',
               style_function=lambda f: {'color': '#52514e', 'weight': 1, 'fillOpacity': 0},
               tooltip=folium.GeoJsonTooltip(['name'], labels=False)).add_to(m)

# One clustered layer per water source (the colored dot appears in the layer control)
for source, color in WATER_COLORS.items():
    subset = df[df['water_source'] == source]
    layer = folium.FeatureGroup(
        name=f'<span style="color:{color}">&#9679;</span> {source} ({len(subset)})')
    cluster = plugins.MarkerCluster(options={'disableClusteringAtZoom': 16}).add_to(layer)
    for _, row in subset.iterrows():
        folium.CircleMarker(
            location=(row['lat'], row['lon']), radius=6,
            color='white', weight=1.5, fill=True, fill_color=color, fill_opacity=0.9,
            popup=folium.Popup(popup_html(row), max_width=300),
            tooltip=row['name'] if pd.notna(row['name']) else source).add_to(cluster)
    layer.add_to(m)

# Heatmap (switched off by default)
plugins.HeatMap(df[['lat', 'lon']].values.tolist(), name='Heatmap',
                radius=18, blur=15, show=False).add_to(m)

# Map controls
folium.LayerControl(collapsed=False).add_to(m)
plugins.Fullscreen().add_to(m)
plugins.MiniMap(toggle_display=True).add_to(m)
plugins.MeasureControl(primary_length_unit='meters',
                       primary_area_unit='sqmeters').add_to(m)
m

### Fountain density per quarter

In [ ]:
# Choropleth map: fountains per km² (quantile classes, so the few very dense
# quarters in the old town do not dominate the color scale)
for feature in geojson['features']:
    row = df_quarters.loc[feature['properties']['name']]
    feature['properties'].update({'fountains': int(row['fountains']),
                                  'land_km2': round(row['land_km2'], 2),
                                  'per_km2': round(row['per_km2'], 1)})

m2 = folium.Map(location=[df.lat.mean(), df.lon.mean()], zoom_start=12,
                tiles=swisstopo_url('ch.swisstopo.swisstlm3d-karte-grau', 'png'),
                attr=SWISSTOPO_ATTR, control_scale=True)

choropleth = folium.Choropleth(
    geo_data=geojson, data=df_quarters.reset_index(), columns=['quarter', 'per_km2'],
    key_on='feature.properties.name', fill_color='Blues',
    bins=list(df_quarters['per_km2'].quantile([0, 0.2, 0.4, 0.6, 0.8, 1]).round(1)),
    fill_opacity=0.8, line_color='white', line_weight=1.5,
    legend_name='Drinking fountains per km²').add_to(m2)

folium.GeoJsonTooltip(
    fields=['name', 'fountains', 'land_km2', 'per_km2'],
    aliases=['Quarter', 'Fountains', 'Land area (km²)', 'Fountains per km²']
).add_to(choropleth.geojson)
m2

In [ ]:
# The same values as a ranking
fig = px.bar(df_quarters.sort_values('per_km2').reset_index(), x='per_km2', y='quarter',
             orientation='h', hover_data={'fountains': True, 'land_km2': ':.2f'},
             labels={'per_km2': 'Fountains per km² (land area)', 'quarter': '',
                     'fountains': 'Fountains', 'land_km2': 'Land area (km²)'},
             title='Fountain density per city quarter',
             template='simple_white', height=750)
fig.update_traces(marker_color='#2a78d6')
fig.show()

### How far is it to the nearest fountain?

In [ ]:
# Distance raster in LV95 coordinates
fig, ax = plt.subplots(figsize=(10, 9))
extent = [xs[0] - cell / 2, xs[-1] + cell / 2, ys[0] - cell / 2, ys[-1] + cell / 2]
im = ax.imshow(np.ma.masked_invalid(dist_grid), origin='lower', extent=extent,
               cmap='Blues', vmin=0, vmax=600)

# Lake, quarter boundaries and fountains
for p in lake_paths:
    ax.fill(p.vertices[:, 0], p.vertices[:, 1], color='#d9d8d4', zorder=0)
ax.text(2684600, 1244200, 'Lake Zürich', color='#52514e', fontsize=10, style='italic')
for name, ps in paths.items():
    for p in ps:
        ax.plot(p.vertices[:, 0], p.vertices[:, 1], color='#52514e', linewidth=0.6)
ax.scatter(points[:, 0], points[:, 1], s=4, color='#eb6834', label='Drinking fountain')

fig.colorbar(im, ax=ax, shrink=0.7, extend='max',
             label='Distance to the nearest fountain (m)')
ax.set_title('How far is it to the nearest drinking fountain?', loc='left', fontweight='bold')
ax.set_xlabel('East (LV95, m)')
ax.set_ylabel('North (LV95, m)')
ax.ticklabel_format(style='plain')
ax.set_xlim(extent[:2])
ax.set_ylim(extent[2:])
ax.set_aspect('equal')
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

In [ ]:
# Share of the city area within a given distance of a fountain
distances = np.arange(0, 1001, 10)
df_cov = pd.DataFrame({'distance_m': distances,
                       'share': [(d <= r).mean() * 100 for r in distances]})

fig = px.area(df_cov, x='distance_m', y='share',
              labels={'distance_m': 'Distance to the nearest fountain (m)',
                      'share': 'Share of the city area (%)'},
              title='Share of the city within walking distance of a fountain',
              template='simple_white', height=400)
fig.update_traces(line_color='#2a78d6')
for r in [200, 500]:
    share = (d <= r).mean() * 100
    fig.add_annotation(x=r, y=share, text=f'{share:.0f}% within {r} m',
                       showarrow=True, arrowhead=2, ax=40, ay=30)
fig.update_yaxes(range=[0, 105])
fig.show()

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')